[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/decimal-labs/decimalai-python/blob/main/examples/measure-a-skill/measure_a_skill.ipynb)

# Don't trust our number

DecimalAI's registry ranks agent skills by a **measured** with-vs-without benchmark
— not stars, not downloads. This notebook hands you the evidence behind one of those
numbers so you can decide whether it means anything.

**Runtime → Run all.** Nothing here asks you to sign up or install anything.

| | needs | you get |
|---|---|---|
| **You are here** | nothing | the claim, the test suite, the per-case transcripts, the safety scan, and what we refuse to show you |
| Next notebook | one free Google AI Studio key | re-run the benchmark yourself, blind-judged, on your model |
| After that | a DecimalAI account | routing and traces for your own agent |

Stopping after this cell block is a perfectly good outcome. You will have audited a
vendor without giving it anything.

## Setup — one helper, no installs

Colab already ships `requests` and `PyYAML`, so there is nothing to `pip install`.

The retry helper is not boilerplate. Anonymous traffic to the registry is rate-limited
at the edge, and a cold cache on the measured-skill listing can take a few seconds — a
notebook whose entire pitch is *re-run it yourself* cannot fall over on the fourth
request.

In [ ]:
import json, time, textwrap
import requests, yaml

API = "https://api.decimal.ai/api/v1"
MANIFEST_URL = "https://raw.githubusercontent.com/decimal-labs/decimalai-python/main/examples/measure-a-skill/manifest.yaml"


def get(path, **params):
    """GET with backoff. Distinguishes 'rate limited' from 'actually missing'."""
    url = path if path.startswith("http") else f"{API}{path}"
    delay = 1.0
    for attempt in range(5):
        try:
            r = requests.get(url, params=params or None, timeout=30)
        except requests.exceptions.RequestException as exc:
            # A dropped connection is the same event as a 503, one layer down.
            # Observed: one read timeout against app.decimal.ai ended this
            # notebook in a 60-line urllib3 traceback at the last cell.
            if attempt == 4:
                raise RuntimeError(f"{type(exc).__name__} on {url}") from None
            time.sleep(delay)
            delay *= 2
            continue
        if r.status_code == 200:
            return r.json() if "json" in r.headers.get("content-type", "") else r.text
        if r.status_code in (429, 500, 502, 503, 504):
            time.sleep(delay)
            delay *= 2
            continue
        raise RuntimeError(f"{r.status_code} on {url}")
    raise RuntimeError(f"gave up after 5 attempts: {url}")


def wrap(text, width=88, indent="    "):
    return textwrap.indent(textwrap.fill(str(text), width), indent)


# The manifest lives on `main` so a stale figure is a one-line YAML edit rather
# than notebook surgery. The embedded copy below is the fallback: raw.github can
# be briefly unavailable, and a reader who hits that moment should get a working
# notebook and a note — not a KeyError on a 404 page parsed as YAML.
FALLBACK = {
    # demo_case is case-22 ON PURPOSE — the case where this skill LOSES. Keep the
    # two copies in step; a fallback that quietly disagrees with main is how a
    # notebook starts telling a different story than its manifest.
    "demo_skill": {"slug": "flsa-exemption-test", "author_display_name": "ashenpivot",
                   "source_type": "platform", "version": 1, "demo_case": "case-22",
                   "contested_case": "case-01"},
    "gates": {"min_delta_pts": 20, "min_cases": 20,
              "require_grading_method": "judged"},
    "registry_disclosure": {"measured_public_skills": 1429, "graded_cases": 30506,
                            "cases_withheld_pct": 12.6, "platform_cases_withheld_pct": 56.6,
                            "github_import_cases_withheld_pct": 0.7, "as_of": "2026-08-10"},
}

try:
    M = yaml.safe_load(requests.get(MANIFEST_URL, timeout=10).text)
    if not isinstance(M, dict) or "demo_skill" not in M:
        raise ValueError("manifest did not parse as expected")
    print("manifest: fetched from main")
except Exception as exc:
    M = FALLBACK
    print(f"manifest: using the embedded copy ({type(exc).__name__}) — "
          "figures may lag what is on main")

DEMO, GATES = M["demo_skill"], M["gates"]
print("demo skill:", DEMO["slug"])

## What survives an evidence gate

Below is the quality bar this notebook applies, run against the top of the leaderboard
in front of you. The point is not the survivors — it is **how few there are**, and that
you can watch the filter run rather than take our word for the shortlist.

In [ ]:
rows = get("/registry/skills", measured="only", sort="lift", limit=100)
items = rows.get("items", rows if isinstance(rows, list) else [])

def passes(s):
    # Every gate field lives under benchmark_summary, never at the top level —
    # reading them flat silently yields None and passes everything.
    b = s.get("benchmark_summary") or {}
    return (
        b.get("grading_method") == GATES["require_grading_method"]
        and (b.get("total_cases") or 0) >= GATES["min_cases"]
        and (b.get("pass_rate_delta_pts") or 0) >= GATES["min_delta_pts"]
    )

kept = [s for s in items if passes(s)]
print(f"{len(items)} skills in → {len(kept)} survive the gate\n")
for s in kept[:8]:
    b = s["benchmark_summary"]
    print(f"  {b['pass_rate_delta_pts']:+6.2f} pts  {b['total_cases']:>3} cases  "
          f"{s.get('source_type','?'):<14} {s['url_slug']}")

print("\nA high delta is not the same as a legible one, which is why the demo skill")
print("below is pinned by name rather than taken from the top of this list.")

## The claim, and every qualifier attached to it

In [ ]:
skill = get(f"/registry/skills/{DEMO['slug']}")
b = skill["benchmark_summary"]

# Identity, not just existence. A slug that still resolves is not the skill you
# meant: one slug in this registry still returns 200 and is now a medical-skills
# import. If these pins fail we use the spare rather than describe the wrong thing.
assert skill["author_display_name"] == DEMO["author_display_name"], "author changed"
assert skill["source_type"] == DEMO["source_type"], "source_type changed"

with_pct = 100.0 * b["passed_cases"] / b["total_cases"]
without_pct = with_pct - b["pass_rate_delta_pts"]

print(f"{skill['url_slug']}  ({skill['skill_badge']}, {skill['source_type']})\n")
print(f"  lift            {b['pass_rate_delta_pts']:+.2f} pts")
print(f"  without skill   {without_pct:.1f}% of {b['total_cases']} cases")
print(f"  with skill      {with_pct:.1f}%")
print(f"  cases passed    {b['passed_cases']} with the skill, "
      f"{round(b['total_cases'] * without_pct / 100)} without")
print(f"  graded by       {b.get('judge_model')}  ({b.get('grading_method')})")
print(f"  safety scan     {skill.get('safety_status')}")

print("\n--- the part most scorecards leave off ---")
print(wrap(
    f"The with-skill arm still fails {b['total_cases'] - b['passed_cases']} of "
    f"{b['total_cases']} cases. This skill is not 'solved' — it moved the needle "
    f"{b['pass_rate_delta_pts']:+.1f} points and left plenty on the table. A registry "
    "that only ever showed you the ceiling would be less useful, not more."))
print()
print(wrap(
    "Lift is model-relative. It was measured on "
    f"{b.get('benchmark_model')}: it says this skill supplies knowledge THAT model "
    "lacked, not a universal constant. A stronger base model may need it less."))

## Check our arithmetic before you believe it

This recomputes the headline from the per-case rows in the same payload. It is a
genuine check on one artifact: if the summary and the cases disagree, the summary is
not describing this run.

Deliberately *not* a cross-check against the published eval suite. That comparison
fails on a large fraction of this registry — and the reason is the subject of the next
cell but one, not something to bury here.

In [ ]:
run = get(f"/registry/skills/{DEMO['slug']}/benchmark")["latest_run"]
res = run["results"]

PASSING = {"flip_to_pass", "pass_kept"}
recomputed_pass = sum(1 for r in res if r["outcome"] in PASSING)

assert len(res) == b["total_cases"], "case count disagrees with the summary"
assert recomputed_pass == b["passed_cases"], "pass count disagrees with the summary"

outcomes = {}
for r in res:
    outcomes[r["outcome"]] = outcomes.get(r["outcome"], 0) + 1

print(f"{len(res)} case rows; recomputed {recomputed_pass} passing — matches the headline.\n")
for k, v in sorted(outcomes.items(), key=lambda kv: -kv[1]):
    print(f"  {v:>3}  {k}")
print(f"\n  run version v{run['version_number']}   is_latest={run['is_latest_version']}")

## The case where our own skill loses

A vendor demo picks the case that flatters it. This one picks a case where the skill
**makes the answer worse** — and where our own benchmark grades it as a failure.

That is a deliberate choice, and it is the honest one. While building the next notebook
we found that this suite contradicts itself: the case that most flatters this skill
(`case-01`, a software specialist at $80/hour) is the *same fact pattern* as the case
below — an hourly computer employee well above the $27.63/hr threshold that
29 CFR 541.400(b) exempts. If the case below is right, `case-01` is wrong, and there the
"wrong" no-skill answer was the correct one all along.

We could have shipped `case-01` and it would have looked better. A notebook whose whole
argument is *check our work* cannot lead with the case where our work does not survive
being checked.

In [ ]:
case = next((r for r in res if r["case_name"] == DEMO["demo_case"]), None) \
    or next(r for r in res if r["outcome"] == "fail_kept")

# `outcome` reads `fail_kept` on this case — both arms scored as failing. That
# per-case verdict is the scorecard's own and it is the authoritative one: it is
# read back here exactly as recorded, not re-graded by this notebook. Printed
# rather than hidden, because the expectation rows below disagree with it and you
# should see both.
print(f"{case['case_name']}")
print(f"outcome: {case['outcome']}\n")

# A withheld prompt is a real state, not an error — see the next cell.
if not case.get("case_prompt"):
    print(f"prompt withheld: {case.get('case_prompt_unavailable')}")
else:
    print("PROMPT"); print(wrap(case["case_prompt"])); print()
    print("WITHOUT THE SKILL"); print(wrap(case.get("without_skill_output"))); print()
    print("WITH THE SKILL"); print(wrap(case.get("with_skill_output"))); print()
    missed = [e for e in (case.get("expectation_results") or [])
              if not e.get("passed")]
    if missed:
        print("THE EXPECTATION THE SKILL MISSED")
        print(wrap(missed[0]["expectation"]))
        print()
        print(wrap(
            "Read that again: the arm WITHOUT the skill answered correctly and the arm "
            "WITH it did not. The skill over-applies the salary-basis rule to a category "
            "of worker the regulation carves out. Our benchmark caught it, scored it as "
            "a failure, and it is one of the 11 cases the skill still fails."))

print("\n--- cost, because a skill is never free ---")
agg = run.get("aggregate_metrics") or {}
for key in ("tokens", "duration_ms"):
    blk = agg.get(key) or {}
    if blk.get("delta_pct") is not None:
        print(f"  {key:<12} {blk['delta_pct']:+.1f}% with the skill loaded")

## What this registry refuses to show you

This is the cell no vendor demo includes, and it is the reason to trust the rest.

A benchmark result is only meaningful if the prompt shown beside it is the prompt that
produced it. For a large share of this registry we cannot prove that, so the API now
returns `case_prompt: null` and a reason instead of a plausible-looking pairing.

In [ ]:
d = M["registry_disclosure"]
print(f"As of {d['as_of']}, across {d['measured_public_skills']:,} measured "
      f"public skills / {d['graded_cases']:,} graded cases:\n")
print(f"  {d['cases_withheld_pct']}% of all cases have their prompt withheld")
print(f"  {d['platform_cases_withheld_pct']}% of PLATFORM-authored cases")
print(f"  {d['github_import_cases_withheld_pct']}% of GitHub-import cases")
print()
print(wrap(
    "Read that middle number again: more than half the cases in our own hand-authored "
    "skills cannot currently prove which prompt they were graded against. Two causes — "
    "an eval suite rewritten in place after a run, and an authoring lane that recorded "
    "one placeholder prompt for every case in a suite. Both are ours. Neither changes "
    "a single recorded output or verdict, but both destroy the pairing, and a pairing "
    "we cannot prove is one we should not display.", 88, "  "))
print()
print(wrap(
    f"The skill above is not in that group — all {b['total_cases']} of its prompts are "
    "served, which is a precondition for appearing in this notebook at all. That is "
    "also why it is not the highest-lift skill on the board.", 88, "  "))

## The thing itself

A skill is a Markdown file. Here is the whole of it — the same bytes the benchmark
loaded, pinned to the measured version rather than to whatever is latest.

In [ ]:
v = DEMO["version"]
body = get(f"https://app.decimal.ai/s/{DEMO['slug']}@{v}/SKILL.md")
print(f"# {DEMO['slug']}@{v} — {len(body):,} bytes\n")
print(body)

print("\n" + "-" * 60)
print(wrap(
    "That is the entire product on this axis: a few hundred lines of text, readable "
    "before you install anything, with a measured claim attached and the evidence "
    "behind that claim in the cells above. Nothing is behind a login.", 88, "  "))

## You have not verified anything yet

Everything above is *our* number, *our* test cases, *our* judge. Reading it carefully is
not the same as checking it, and a faithfully-reproduced self-graded exam is still a
self-graded exam.

The next notebook re-runs this exact comparison on **your** model — the same two arms
the scorecard reports, with the skill and without it — under a blind judge you can read,
and prints the result whichever way it falls. It needs one free Google AI Studio key
(no card, ~20 seconds).

Or skip us entirely and take the file: `pip install decimalai` then
`decimalai skills pull flsa-exemption-test --out .claude/skills/` — anonymous,
no account, and your agent picks it up from disk.